In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/data/train_cleaned.csv')
test = pd.read_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/data/test_cleaned.csv')

# CREATE INTERACTION FEATURE
train['price_x_store_age'] = train['product_price'] * train['store_age_years']
test['price_x_store_age'] = test['product_price'] * test['store_age_years']

# Prepare data
X_train = train.drop(['id', 'product_code', 'store_code', 'total_sales'], axis=1)
y_train = train['total_sales']
X_test = test.drop(['id', 'product_code', 'store_code'], axis=1)

# Load preprocessor
with open('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/models/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

# Transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Data loaded and preprocessed ✓")
print(f"Training data shape: {X_train_processed.shape}")
print(f"Target shape: {y_train.shape}")

Data loaded and preprocessed ✓
Training data shape: (6818, 28)
Target shape: (6818,)


In [2]:
# Train validation split
from sklearn.model_selection import train_test_split

# Split: 80% train, 20% validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_processed, y_train, test_size=0.2, random_state=42
)

print(f"Training set: {X_train_split.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

Training set: 5454 samples
Validation set: 1364 samples


In [3]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import time

print("\n" + "="*70)
print("PHASE 6: HYPERPARAMETER TUNING - XGBoost (GridSearchCV)")
print("="*70)

# Define comprehensive parameter grid
param_grid = {
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'max_depth': [3, 5, 7, 9],
    'n_estimators': [100, 150, 200],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5]
}

print(f"\nParameter grid size: {np.prod([len(v) for v in param_grid.values()])} combinations")
print("This will test many combinations with 5-fold CV...")
print("\n⏳ Training in progress (this may take 10-15 minutes)...\n")

# Create base XGBoost model
xgb_base = xgb.XGBRegressor(random_state=42, verbosity=0)

# GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    xgb_base,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,  # Use all CPU cores
    verbose=2   # Print progress
)

# Start timer
start_time = time.time()

# Fit
grid_search.fit(X_train_split, y_train_split)

# End timer
elapsed_time = time.time() - start_time

print(f"\n✓ GridSearchCV completed in {elapsed_time:.2f} seconds")

# Best parameters
print("\n" + "="*70)
print("BEST HYPERPARAMETERS")
print("="*70)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

best_cv_rmse = np.sqrt(-grid_search.best_score_)
print(f"\n✓ Best CV RMSE: {best_cv_rmse:.4f}")
print(f"✓ Best CV R²: {grid_search.best_score_:.4f}")


PHASE 6: HYPERPARAMETER TUNING - XGBoost (GridSearchCV)

Parameter grid size: 1296 combinations
This will test many combinations with 5-fold CV...

⏳ Training in progress (this may take 10-15 minutes)...

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits

✓ GridSearchCV completed in 5939.47 seconds

BEST HYPERPARAMETERS
colsample_bytree: 0.9
learning_rate: 0.05
max_depth: 3
min_child_weight: 1
n_estimators: 150
subsample: 0.8

✓ Best CV RMSE: 1085.9986
✓ Best CV R²: -1179392.8543


In [4]:
# Get best tuned model
best_xgb_tuned = grid_search.best_estimator_

# Evaluate on validation set
y_pred_tuned = best_xgb_tuned.predict(X_val)
rmse_tuned = np.sqrt(mean_squared_error(y_val, y_pred_tuned))
r2_tuned = r2_score(y_val, y_pred_tuned)

print("\n" + "="*70)
print("TUNED XGBoost VALIDATION PERFORMANCE")
print("="*70)
print(f"Validation RMSE: {rmse_tuned:.4f}")
print(f"Validation R²: {r2_tuned:.4f}")

# Compare with original XGBoost
print("\n" + "="*70)
print("BEFORE vs AFTER TUNING")
print("="*70)
comparison_tune = pd.DataFrame({
    'Model': ['XGBoost (Original)', 'XGBoost (Tuned)'],
    'Validation RMSE': [1107.1248, rmse_tuned],
    'Validation R²': [0.583692, r2_tuned]
})
print(comparison_tune.to_string(index=False))

# Calculate improvement
improvement = ((1107.1248 - rmse_tuned) / 1107.1248) * 100
print(f"\n🎯 Improvement: {improvement:.2f}%")


TUNED XGBoost VALIDATION PERFORMANCE
Validation RMSE: 1090.4083
Validation R²: 0.5962

BEFORE vs AFTER TUNING
             Model  Validation RMSE  Validation R²
XGBoost (Original)      1107.124800       0.583692
   XGBoost (Tuned)      1090.408326       0.596169

🎯 Improvement: 1.51%


In [5]:
# Save tuned model
with open('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/models/best_model_tuned.pkl', 'wb') as f:
    pickle.dump(best_xgb_tuned, f)

print("\n✓ Tuned model saved to models/best_model_tuned.pkl")


✓ Tuned model saved to models/best_model_tuned.pkl


In [6]:
print("\n" + "="*70)
print("PHASE 7: FINAL PREDICTIONS & SUBMISSION")
print("="*70)

# Use tuned model to predict on test set
y_test_pred = best_xgb_tuned.predict(X_test_processed)

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'total_sales': y_test_pred
})

# Save to CSV
submission.to_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/submission/final_submission.csv', index=False)

print("\n✓ Submission file created: submissions/final_submission.csv")
print(f"\nSubmission Preview:")
print(submission.head(10))
print(f"\nTotal predictions: {len(submission)}")
print(f"Sales range: {submission['total_sales'].min():.2f} - {submission['total_sales'].max():.2f}")


PHASE 7: FINAL PREDICTIONS & SUBMISSION

✓ Submission file created: submissions/final_submission.csv

Submission Preview:
          id  total_sales
0  row_00009  2652.708496
1  row_00015  4477.582520
2  row_00019  3076.480225
3  row_00020  3165.681152
4  row_00023  2394.310547
5  row_00026   521.528320
6  row_00027  3644.262451
7  row_00033  3226.233887
8  row_00034   561.003479
9  row_00037  2005.666992

Total predictions: 1705
Sales range: -229.51 - 6581.84


In [7]:
# RMSE on validation set (what we've been using)
y_val_pred = best_xgb_tuned.predict(X_val)
rmse_val_final = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2_val_final = r2_score(y_val, y_val_pred)

print("="*70)
print("FINAL MODEL PERFORMANCE - VALIDATION SET")
print("="*70)
print(f"Validation RMSE: {rmse_val_final:.4f}")
print(f"Validation R²: {r2_val_final:.4f}")
print(f"Validation MAE: {np.mean(np.abs(y_val - y_val_pred)):.4f}")

FINAL MODEL PERFORMANCE - VALIDATION SET
Validation RMSE: 1090.4083
Validation R²: 0.5962
Validation MAE: 794.2626
